# 01 · Data understanding
**Steps 1–3:** business problem, dataset + ER diagram, loading the raw data.

## Business problem
An e-commerce platform stores orders, products, sellers, payments, deliveries and reviews
across 9 related tables, which makes performance and customer behaviour hard to see in raw form.
The goal is to turn that data into insights on sales, customers, products, sellers, payments,
delivery performance and customer satisfaction.

## Questions this analysis answers
1. How are revenue, orders and average order value trending, and which categories/states drive them?
2. Who are our customers, how much do they spend and how many come back?
3. Which sellers and products perform best, and how are they rated?
4. How fast are deliveries, how often are they late, and where?
5. Does late delivery hurt review scores? Does spend differ by category? Is payment method linked to order outcome?

In [ ]:
import sys, warnings
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from src import config
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
sns.set_theme(style="whitegrid", palette="viridis")

# Where the raw CSVs live. To rehearse on generated test data instead, use:
#   RAW_DIR = config.SYNTHETIC_DIR   (run `python scripts/make_synthetic_data.py` first)
RAW_DIR = config.RAW_DIR

## Load all 9 CSV files with Pandas

In [ ]:
from src.data_loader import load_raw_tables, describe_tables

raw = load_raw_tables(RAW_DIR)
display(describe_tables(raw)[["table", "rows", "columns"]])

## Shape, columns and data types of each table

In [ ]:
for name, df in raw.items():
    print(f"\n=== {name}: {df.shape[0]:,} rows x {df.shape[1]} columns ===")
    display(df.dtypes.rename("dtype").to_frame().T)
    display(df.head(3))

## Keys and relationships (ER diagram)
The ER diagram is in `docs/er_diagram.md`; the full data dictionary is in `docs/data_dictionary.md`.

In [ ]:
pk = pd.DataFrame([(t, ", ".join(c)) for t, c in config.PRIMARY_KEYS.items()],
                  columns=["table", "primary_key"])
fk = pd.DataFrame(config.FOREIGN_KEYS,
                  columns=["child_table", "child_column", "parent_table", "parent_column"])
display(pk)
display(fk)

In [ ]:
# Quick sanity check of the relationships on the raw data (share of child rows with a parent)
rows = []
for child, ccol, parent, pcol in config.FOREIGN_KEYS:
    match = raw[child][ccol].isin(set(raw[parent][pcol])).mean() * 100
    rows.append({"relationship": f"{child}.{ccol} -> {parent}.{pcol}", "pct_rows_with_parent": match})
display(pd.DataFrame(rows))